# 02 Predictions And External Evaluation

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = 'revision/colab-pipeline'
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    _run_setup('git fetch origin')
    current_branch = subprocess.check_output(
        'git branch --show-current',
        shell=True,
        text=True,
    ).strip()
    if current_branch != BRANCH:
        _run_setup(f'git checkout {BRANCH}')
    _run_setup(f'git pull --ff-only --autostash origin {BRANCH}')
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        _run_setup('git fetch origin')
        _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', check=False)
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Install Base Dependencies

This notebook can run directly after opening in Colab. These packages cover data loading, feature extraction, metrics, and artifact checks.

In [ ]:
INSTALL_BASE_DEPS = True
if INSTALL_BASE_DEPS:
    !python --version
    !pip install -q numpy==1.26.4 scipy==1.11.4 pandas scikit-learn threadpoolctl tqdm wfdb joblib matplotlib seaborn packaging neurokit2 iterative-stratification thop
else:
    print('Skipping base dependency install.')


## Install Model Dependencies

In [ ]:
INSTALL_MODEL_DEPS = True
if INSTALL_MODEL_DEPS:
    import importlib.util
    import subprocess
    import sys

    explicit_wheel_dirs = [
        Path('/content/drive/MyDrive/mamba_wheels_py312'),
        DRIVE_ROOT / 'mamba_wheels_py312',
    ]

    def find_wheels():
        for wheel_dir in explicit_wheel_dirs:
            if not wheel_dir.exists():
                continue
            wheels = sorted(wheel_dir.glob('*.whl'))
            causal = [p for p in wheels if 'causal_conv1d' in p.name]
            mamba = [p for p in wheels if 'mamba_ssm' in p.name]
            if causal and mamba:
                return causal[0], mamba[0], wheel_dir

        search_roots = [DRIVE_ROOT, Path('/content/drive/MyDrive')]
        seen = set()
        for root in search_roots:
            if not root.exists() or root in seen:
                continue
            seen.add(root)
            print(f'Searching for Mamba wheels under: {root}')
            causal = sorted(root.rglob('causal_conv1d*.whl'))
            mamba = sorted(root.rglob('mamba_ssm*.whl'))
            if causal and mamba:
                return causal[0], mamba[0], causal[0].parent
        return None, None, None

    if importlib.util.find_spec('mamba_ssm') is not None:
        print('mamba-ssm already installed.')
    else:
        causal_wheel, mamba_wheel, wheel_dir = find_wheels()
        if causal_wheel is None or mamba_wheel is None:
            print('No local Mamba wheels found.')
            print('Expected directory examples:')
            for p in explicit_wheel_dirs:
                print('-', p)
            raise FileNotFoundError(
                'Missing causal_conv1d*.whl and/or mamba_ssm*.whl on Drive. '
                'Do not pip-build mamba-ssm on Colab; upload/copy prebuilt wheels first.'
            )

        print(f'Installing Mamba wheels from: {wheel_dir}')
        print('causal-conv1d:', causal_wheel.name)
        print('mamba-ssm   :', mamba_wheel.name)
        subprocess.run([sys.executable, '-m', 'pip', 'install', str(causal_wheel), '-q'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', str(mamba_wheel), '-q'], check=True)
        print('Installed causal-conv1d and mamba-ssm from local wheels.')

    import mamba_ssm
    print('mamba_ssm import OK:', mamba_ssm.__file__)
else:
    print('Skipping mamba-ssm install. Ensure it is already available before prediction generation.')


## Prediction Contract

Every downstream metric script expects NPZ files with `y_true` and `y_prob`, both shaped `(N, C)`. Store prediction files under `reports/revision/predictions/`.

In [ ]:
from pathlib import Path
import numpy as np

pred_dir = Path('reports/revision/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(pred_dir.glob('*.npz')):
    data = np.load(path, allow_pickle=True)
    keys = sorted(data.files)
    print(path, keys)
    if {'y_true', 'y_prob'}.issubset(keys):
        print('  y_true:', data['y_true'].shape, 'y_prob:', data['y_prob'].shape)


## Generate OOF Predictions

In [ ]:
RUN_OOF_EXPORT = True
BATCH_SIZE = 32
DEBUG_LIMIT_RECORDS = 0

command = (
    'python scripts/revision/01_generate_predictions.py '
    f'--dataset oof --checkpoint-kind best --batch-size {BATCH_SIZE}'
)
if DEBUG_LIMIT_RECORDS:
    command += f' --limit-records {DEBUG_LIMIT_RECORDS}'

if RUN_OOF_EXPORT:
    run(command)
else:
    print(f'OOF export disabled. Set RUN_OOF_EXPORT=True to execute: {command}')


## Verify OOF Prediction Outputs

In [ ]:
expected = [
    Path('reports/revision/predictions/oof_full_predictions.npz'),
    Path('reports/revision/predictions/oof_full_slice_predictions.npz'),
    Path('reports/revision/metrics/oof_full_prediction_summary.json'),
]

for path in expected:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

pred_path = expected[0]
if pred_path.exists():
    data = np.load(pred_path, allow_pickle=True)
    print('keys:', sorted(data.files))
    print('y_true:', data['y_true'].shape)
    print('y_prob:', data['y_prob'].shape)
    print('classes:', list(data['class_names'])[:5], '...', len(data['class_names']))


## External Prediction Commands

In [ ]:
print('PTB-XL and CPSC/Georgia prediction exporters are the next implementation step.')
print('Expected future outputs:')
print('- reports/revision/predictions/ptbxl_full_predictions.npz')
print('- reports/revision/predictions/cpsc_full_predictions.npz')


## Re-run Inventory

In [ ]:
!python scripts/revision/05_artifact_inventory.py
